# Struktura szumu wskazań poza zdarzeniami

Jak wygląda szum wskazań wagi poza zdarzeniami, od najkrótszej skali czasu do najdłuższej: skoki między kolejnymi odczytami, powiązanie odczytów w czasie (autokorelacja) i udział wahań o różnych okresach (widmo). Po co: metody wyliczania masy odsypu uśredniają okna przed i po odsypie; uśrednianie zmniejsza szum tylko wtedy, gdy kolejne odczyty są od siebie niezależne. Zbiór: baza `2026-10-18 obiekt`, sesja 9, wariant `CURRENT_VARIANT`, fragmenty bez zdarzeń bez zakłóceń (`dms_analysis.between_events_fragments`); obliczenia w `dms_analysis.noise_structure`.

**Odniesienie „biały szum”:** sztuczne wskazania o tym samym rozrzucie skoków co dane, ale z odczytami całkowicie od siebie niezależnymi, zaokrąglone do działki 20 kg (`white_noise_samples`). Wszędzie, gdzie dane wyglądają inaczej niż biały szum, szum wagi ma strukturę w czasie. Pora dnia wg zegara RPi: noc `NIGHT_HOURS`, dzień `DAY_HOURS`.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd
from IPython.display import display

from dms_analysis.between_events_fragments import prepare_samples, white_noise_samples
from dms_analysis.noise_structure import (
    DETREND_SPAN_S,
    autocorrelation,
    autocorrelation_figure,
    band_summary,
    detrend_figure,
    jump_figure,
    jump_summary,
    jumps,
    residual_windows,
    segments,
    spectrum,
    spectrum_figure,
    white_sigma_kg,
)
from dms_analysis.prepared import load_session
from dms_analysis.signals import CURRENT_VARIANT
from dms_analysis.snapshot import ANALYSIS_SESSION, ANALYSIS_SOURCE

DB_PATH = Path("../data/prepared/obiekt-sesja-9.db")
FIGURE_WIDTH = 1200

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

session = load_session(DB_PATH, ANALYSIS_SOURCE, ANALYSIS_SESSION, CURRENT_VARIANT)
samples = prepare_samples(session)

jump_table = jumps(samples)
white = white_noise_samples(samples, white_sigma_kg(jump_table))

## 1. Skoki między kolejnymi odczytami

O ile działek (20 kg) zmienia się wskazanie z odczytu na odczyt (co ok. 1 s). Tabela: procent wszystkich par odczytów. Oś Y wykresu jest logarytmiczna, żeby było widać rzadkie duże skoki.

**Jak czytać:** biały szum o tym samym rozrzucie miałby dużo skoków o ±1 i prawie zero skoków większych niż ±3. Jeśli dane mają więcej „0” i jednocześnie więcej dużych skoków, to szum nie jest równomiernym drganiem, tylko długie spokojne okresy przerywa od czasu do czasu wyraźny skok.

In [ ]:
jump_table_summary = jump_summary(jump_table, jumps(white))
display(jump_table_summary.round(2))

jump_figure(jump_table_summary, FIGURE_WIDTH).show()

## 2. Autokorelacja: czy kolejne odczyty są od siebie niezależne

Z każdego odcinka 60 s odejmujemy dopasowaną w nim prostą (wolny trend), zostają reszty. Autokorelacja dla przesunięcia k mówi, na ile reszta teraz jest podobna do reszty k sekund później: 1 = taka sama, 0 = niezależna. Dla każdego odcinka liczona osobno i uśredniona (odcinki ze stałym wskazaniem, bez reszt, pominięte).

Przykład poniżej: górny wykres = wskazania i odejmowana prosta, dolny = reszty, z których liczona jest autokorelacja.

**Jak czytać:** biały szum spada do zera od razu (przesunięcie 1 s). Jeśli dane spadają do zera dopiero po kilku–kilkunastu sekundach, to tyle sekund trwa jedno „drgnięcie” i odczyty z tego czasu niosą w praktyce jedną informację — średnia z okna N s ma wtedy większy szum, niż wynikałoby z N niezależnych odczytów. Ujemne wartości przy dużych przesunięciach to częściowo skutek odejmowania prostej (biały szum też lekko schodzi poniżej zera).

In [ ]:
detrend_figure(samples, fragment=3, start_min=0, minutes=10, width=FIGURE_WIDTH).show()

In [ ]:
correlations = autocorrelation(residual_windows(samples, DETREND_SPAN_S), residual_windows(white, DETREND_SPAN_S))
display(correlations.loc[[1, 2, 3, 5, 10, 20, 30]].round(2).rename_axis("przesunięcie [s]"))

autocorrelation_figure(correlations, FIGURE_WIDTH).show()

## 3. Widmo: wahania o jakim okresie dominują

Ciągłe kawałki fragmentów pocięte na odcinki 20 min (z zakładką 50%), z każdego odjęta prosta, potem moc wahań w zależności od okresu (metoda Welcha, średnia z odcinków). Oś X = okres wahania (od 2 s do 20 min), obie osie logarytmiczne.

**Jak czytać wykres:** biały szum to pozioma linia (wszystkie okresy po równo). Linia rosnąca w prawo = im wolniejsze wahania, tym silniejsze (dryf, błądzenie). Wyraźny „garb” albo szpic przy jakimś okresie = powtarzające się wahanie o tym okresie.

**Tabela** jest czytelniejszą wersją wykresu: ile kg odchylenia standardowego pochodzi z wahań o okresie w danym przedziale. Przedziały 10–30 s i 30–60 s to skala okien przy odsypie i samego odsypu; wahania dłuższe niż kilka minut usuwa korekcja trendem, krótsze niż kilka sekund — uśrednianie.

In [ ]:
spectrum_table = spectrum(segments(samples), segments(white))
display(band_summary(spectrum_table).round(1).rename_axis("okres wahania"))

spectrum_figure(spectrum_table, FIGURE_WIDTH).show()